# 8.4 能依情境切換風格嗎？


同一位老師教小朋友可以多舉例，幫同事查數字時則可以直接給結論。我們也希望一個模型根據要求切換，而不是只能永遠簡短或永遠多話。問題是：如果訓練資料恰好「算術全部短答、生活全部長答」，模型可能學到題材決定風格，而不是學到使用者的要求。

先讀[8.3風格如何進入訓練資料](https://birdhackor.github.io/tiny-perceptron-vlm/8.3.html)。那裡讓相同問題有兩種內容一致的寫法；本節再把要求也放進輸入。這稱為條件式風格：條件就是模型當下看得見的「簡短」或「生動」字樣。它不是看不見的作者意圖，也不是資料檔外面一個模型無法讀到的欄位。

讓兩道算術題都同時出現在兩種風格下，讀者就能看出資料是否有必要的對照。這像教老師不要看到加法就自動省略解釋，而是先讀家長要求。下面採用人工規則產生四筆資料；規則只負責把例子寫齊，還不是神經模型已學會切換的證據。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
records = []
for question, fact in [("1+1=?", "2"), ("2+2=?", "4")]:
    for style in ["簡短", "生動"]:
        reply = fact if style == "簡短" else fact + "；把兩組物件合起來數就知道了。"
        records.append({"提問": f"風格={style}；{question}", "回答": reply})
print("筆數", len(records))
for row in records:
    print(row["提問"], "→", row["回答"])

輸入有兩對`question`與`fact`，分別保存題目與事實答案。外層迴圈換題目，內層迴圈換風格，因此每題會走兩次。`reply`那一行是條件判斷：簡短就直接用數字，其餘情況加一句解釋。`append`把新字典加入清單。輸出先印`筆數 4`，再印1+1的簡短與生動版、2+2的簡短與生動版；數字答案始終是2或4。

這四筆覆蓋了「兩題乘兩種風格」的組合。如果只留下第一題短答、第二題生動，題目和風格一起變，模型可能直接記住1+1要短答。要確認它讀了條件，測試時用未見過的3+3，分別要求兩種風格，核對事實都為6，而寫法隨要求改變。資料範例與成效測量是兩個不同步驟，這段程式完成前者。

還可以把同一題的兩種要求交替排列，避免資料前半全簡短、後半全生動，讓後面幾步訓練把前面的風格蓋過去。這個排列安排讓每一個要求都持續搭配相應目標。若使用者要求「簡短」卻得到長版，要先回查輸入是否真的保存了風格字樣，再判斷模型是否讀懂。

練習把`["簡短", "生動"]`改成只含`["簡短"]`，先預測筆數，再重跑。你會得到2筆，而且完全沒有「同一題改要求」的訓練對照。若恢復兩種風格後想加第三種，先替它寫清楚目標回答，不要只是增加一個名字，卻讓三個條件都指向同一句話。
